# example_pipeline_demo — demo scenario generator for `02_pipeline`

Use this notebook after `00_env_config` to create deterministic demo source tables in the configured `source_lakehouse`. It does **not** run the pipeline guardrails or write unified targets itself. Instead, it prepares small `demo_` source tables and governance-approved DQ rule metadata that the real `02_pipeline` template can read to demonstrate the happy path plus schema, DQ, freshness, and profile behavior guardrails. `02_pipeline` writes the catalogue, lineage, target table, and pipeline run evidence.

Safety notes:

- Writes only tables whose names start with `demo_`.
- Writes to the configured `source_lakehouse` through `write_lakehouse_table(..., CONFIG, ENV, "source", ..., schema=SOURCE_SCHEMA)`.
- Uses overwrite mode so the notebook is safe to rerun.
- Seeds active governance-approved DQ rules only for `demo_` table names in `METADATA_GUARDRAIL_RULES`, routed through the configured metadata lakehouse.
- Does not create, replace, or clean non-demo business tables.
- Keep `example_dq_rule_smoke_test.ipynb` for isolated DQ rule coverage; this notebook is only for end-to-end pipeline scenarios.


## 1. Run `00_env_config`

Run the shared environment notebook first so `CONFIG`, `ENV`, and configured Lakehouse targets are available.


In [1]:
%run 00_env_config


StatementMeta(, , -1, SessionStarting, , SessionStarting, True)

## 2. Import required functions

The generator uses FabricOps IO helpers only to persist source scenario tables and seed demo-scoped DQ rule metadata. The real pipeline logic remains in `02_pipeline`.


In [ ]:
import json
from datetime import date, datetime, timedelta

from pyspark.sql.types import DateType, DoubleType, LongType, StringType, StructField, StructType, TimestampType

from fabricops_kit import write_lakehouse_table
from fabricops_kit.config import _current_audit_timestamp
from fabricops_kit.governance_review import _get_governance_metadata_schemas


StatementMeta(, , -1, Waiting, , Waiting, True)

## 3. Define deterministic smoke scenarios

Each source table is prefixed with `demo_` and represents a scenario that can be selected in `02_pipeline` by editing source or target table dictionaries after DataFrames exist.

- `demo_src_orders_happy` and `demo_src_customers_happy` are the default happy-path pair. They join on `customer_id` so `02_pipeline` can demonstrate many-source transformation and multiple targets.
- `demo_src_orders_schema_drift` intentionally omits `order_amount` and adds `promo_code`, so the schema guardrail should fail before the target write.
- `demo_src_orders_dq_issue` keeps the expected schema but includes null, negative, invalid-status, and duplicate-order records to trigger the governance-approved-rule DQ guardrail.
- `demo_src_orders_stale` keeps the expected schema but uses old `order_date` / `ingestion_ts` values to trigger the freshness guardrail.
- `demo_src_orders_reload_a` and `demo_src_orders_reload_b` support rerunning `02_pipeline` with different row counts and max timestamps to demonstrate static versus changing profile behavior.


In [ ]:
DEMO_PREFIX = "demo_"
DEMO_PIPELINE_DATASET_NAME = "demo_pipeline_orders"

SOURCE_SCENARIO_TABLES = [
    "demo_src_orders_happy",
    "demo_src_customers_happy",
    "demo_src_orders_schema_drift",
    "demo_src_orders_dq_issue",
    "demo_src_orders_stale",
    "demo_src_orders_reload_a",
    "demo_src_orders_reload_b",
]

for table_name in SOURCE_SCENARIO_TABLES:
    if not table_name.startswith(DEMO_PREFIX):
        raise ValueError(f"Refusing to write non-demo table: {table_name}")

orders_schema = StructType(
    [
        StructField("order_id", LongType(), True),
        StructField("customer_id", LongType(), True),
        StructField("order_date", DateType(), True),
        StructField("ingestion_ts", TimestampType(), True),
        StructField("status", StringType(), True),
        StructField("order_amount", DoubleType(), True),
        StructField("country_code", StringType(), True),
    ]
)

customers_schema = StructType(
    [
        StructField("customer_id", LongType(), True),
        StructField("customer_name", StringType(), True),
        StructField("customer_segment", StringType(), True),
        StructField("customer_country_code", StringType(), True),
        StructField("effective_date", DateType(), True),
        StructField("ingestion_ts", TimestampType(), True),
    ]
)

schema_drift_schema = StructType(
    [
        StructField("order_id", LongType(), True),
        StructField("customer_id", LongType(), True),
        StructField("order_date", DateType(), True),
        StructField("ingestion_ts", TimestampType(), True),
        StructField("status", StringType(), True),
        StructField("country_code", StringType(), True),
        StructField("promo_code", StringType(), True),
    ]
)

TODAY = date.today()
FRESH_0 = TODAY
FRESH_1 = TODAY - timedelta(days=1)
STALE = TODAY - timedelta(days=45)


def at_noon(day):
    return datetime(day.year, day.month, day.day, 12, 0, 0)


scenario_frames = {
    "demo_src_orders_happy": spark.createDataFrame(
        [
            (1001, 501, FRESH_0, at_noon(FRESH_0), "new", 19.99, "US"),
            (1002, 502, FRESH_0, at_noon(FRESH_0), "processing", 125.00, "GB"),
            (1003, 503, FRESH_1, at_noon(FRESH_1), "complete", 42.50, "NL"),
        ],
        orders_schema,
    ),
    "demo_src_customers_happy": spark.createDataFrame(
        [
            (501, "Avery Retail", "consumer", "US", FRESH_1, at_noon(FRESH_0)),
            (502, "Beacon Wholesale", "business", "GB", FRESH_1, at_noon(FRESH_0)),
            (503, "Cedar Market", "consumer", "NL", FRESH_1, at_noon(FRESH_0)),
        ],
        customers_schema,
    ),
    "demo_src_orders_schema_drift": spark.createDataFrame(
        [
            (2001, 601, FRESH_0, at_noon(FRESH_0), "new", "US", "WELCOME10"),
            (2002, 602, FRESH_0, at_noon(FRESH_0), "complete", "CA", ""),
        ],
        schema_drift_schema,
    ),
    "demo_src_orders_dq_issue": spark.createDataFrame(
        [
            (None, 701, FRESH_0, at_noon(FRESH_0), "new", 25.00, "US"),
            (3002, 702, FRESH_0, at_noon(FRESH_0), "complete", -5.00, "GB"),
            (3003, 703, FRESH_0, at_noon(FRESH_0), "invalid_status", 10.00, "NL"),
            (3003, 704, FRESH_1, at_noon(FRESH_1), "processing", 11.00, "US"),
        ],
        orders_schema,
    ),
    "demo_src_orders_stale": spark.createDataFrame(
        [
            (4001, 801, STALE, at_noon(STALE), "complete", 75.00, "US"),
            (4002, 802, STALE, at_noon(STALE), "processing", 15.00, "GB"),
        ],
        orders_schema,
    ),
    "demo_src_orders_reload_a": spark.createDataFrame(
        [
            (5001, 901, FRESH_1, at_noon(FRESH_1), "new", 10.00, "US"),
            (5002, 902, FRESH_1, at_noon(FRESH_1), "complete", 20.00, "GB"),
        ],
        orders_schema,
    ),
    "demo_src_orders_reload_b": spark.createDataFrame(
        [
            (5001, 901, FRESH_1, at_noon(FRESH_1), "new", 10.00, "US"),
            (5002, 902, FRESH_1, at_noon(FRESH_1), "complete", 20.00, "GB"),
            (5003, 903, FRESH_0, at_noon(FRESH_0), "processing", 30.00, "NL"),
            (5004, 904, FRESH_0, at_noon(FRESH_0), "complete", 40.00, "CA"),
        ],
        orders_schema,
    ),
}


StatementMeta(, , -1, Waiting, , Waiting, True)

## 4. Create or replace source scenario tables

This cell physically writes each scenario DataFrame to the configured `source_lakehouse`. It uses overwrite mode and refuses to write names that do not start with `demo_`, making the notebook safe to rerun without touching non-demo assets.


In [ ]:
write_results = []
for table_name, dataframe in scenario_frames.items():
    if not table_name.startswith(DEMO_PREFIX):
        raise ValueError(f"Refusing to write non-demo table: {table_name}")
    write_lakehouse_table(
        dataframe,
        CONFIG,
        ENV,
        "source",
        table_name,
        schema="Demo",
        mode="overwrite",
        options={"overwriteSchema": "true"},
    )
    write_results.append({"source_table": table_name, "row_count": dataframe.count(), "write_mode": "overwrite"})

write_results_df = spark.createDataFrame(write_results)
display(write_results_df.orderBy("source_table"))


StatementMeta(, , -1, Waiting, , Waiting, True)

## 5. Seed demo-scoped metadata for current `02_pipeline` and `03_governance` flow

This cell appends schema-valid demo metadata to the currently implemented metadata tables. It uses `METADATA_GUARDRAIL_RULES` for reviewable guardrail intent and `METADATA_GUARDRAIL_RESULTS` only for illustrative runtime evidence. Legacy manual column context/classification and governance review tables are not seeded because the current workflow stores enrichment and review history in `METADATA_ENRICHMENT_RULES` and `METADATA_GUARDRAIL_RULES`.


In [ ]:
now_utc = _current_audit_timestamp(config=CONFIG, drop_microseconds=False)
metadata_schemas = _get_governance_metadata_schemas()
metadata_schema_name = CONFIG.path_config.paths[ENV]["metadata"].schema

DEMO_RULE_TABLES = [
    "demo_src_orders_happy",
    "demo_src_orders_dq_issue",
    "demo_src_orders_stale",
    "demo_src_orders_reload_a",
    "demo_src_orders_reload_b",
    "demo_unified_orders_enriched",
]

DEMO_DQ_RULE_DEFINITIONS = [
    {"rule_type": "not_null", "columns": ["order_id"], "severity": "error", "description": "Demo orders require order_id."},
    {"rule_type": "unique", "columns": ["order_id"], "severity": "error", "description": "Demo orders should not duplicate order_id."},
    {"rule_type": "greater_than_or_equal", "columns": ["order_amount"], "value": 0, "severity": "error", "description": "Demo orders require non-negative order_amount."},
    {"rule_type": "accepted_values", "columns": ["status"], "allowed_values": ["new", "processing", "complete", "cancelled"], "severity": "error", "description": "Demo orders require a known status."},
]

DEMO_GUARDRAIL_RULE_DEFINITIONS = [
    {"guardrail_type": "freshness", "rule_type": "max_age_days", "column_name": "ingestion_ts", "parameters": {"column": "ingestion_ts", "max_age_days": 7}, "severity": "error", "description": "Demo orders should have recent ingestion timestamps."},
    {"guardrail_type": "profile_behavior", "rule_type": "changing_data", "column_name": "ingestion_ts", "parameters": {"profile_mode": "changing_data", "watermark_column": "ingestion_ts"}, "severity": "warning", "description": "Demo orders use changing-data profile behavior."},
]


def metadata_key(table_name, column_name=""):
    return "||".join([str(ENV), DEMO_PIPELINE_DATASET_NAME, table_name, column_name])


def complete_row(table_name, row):
    fields = [field.name for field in metadata_schemas[table_name].fields]
    return {field: row.get(field) for field in fields}


def build_demo_rule_rows(table_name):
    rows = []
    for rule in DEMO_DQ_RULE_DEFINITIONS:
        first_column = list(rule.get("columns", []))[0]
        rule_id = f"demo_pipeline_orders__{table_name}__{rule['rule_type']}"
        parameters = {key: value for key, value in rule.items() if key not in {"rule_type", "severity", "description"}}
        rows.append(build_rule_row(table_name, first_column, "dq", rule["rule_type"], parameters, rule["severity"], rule["description"], rule_id))
    for rule in DEMO_GUARDRAIL_RULE_DEFINITIONS:
        rule_id = f"demo_pipeline_orders__{table_name}__{rule['guardrail_type']}__{rule['rule_type']}"
        rows.append(build_rule_row(table_name, rule["column_name"], rule["guardrail_type"], rule["rule_type"], rule["parameters"], rule["severity"], rule["description"], rule_id))
    return rows


def build_rule_row(table_name, column_name, guardrail_type, rule_type, parameters, severity, description, rule_id):
    return complete_row("METADATA_GUARDRAIL_RULES", {
        "rule_key": metadata_key(table_name, rule_id),
        "rule_id": rule_id,
        "metadata_column_key": metadata_key(table_name, column_name),
        "metadata_table_key": metadata_key(table_name),
        "environment_name": ENV,
        "dataset_name": DEMO_PIPELINE_DATASET_NAME,
        "table_name": table_name,
        "column_name": column_name,
        "guardrail_type": guardrail_type,
        "rule_type": rule_type,
        "rule_parameters_json": json.dumps(parameters, sort_keys=True),
        "severity": severity,
        "description": description,
        "activation_state": "active",
        "is_active": True,
        "review_status": "governance_approved",
        "review_state": "approved",
        "created_by_role": "engineering",
        "author_role": "demo_generator",
        "created_by": "example_pipeline_demo",
        "created_at": now_utc,
        "approved_by": "example_pipeline_demo",
        "approved_at": now_utc,
        "suggestion_json": "{}",
        "action_type": "created",
        "source_notebook_type": "example_pipeline_demo",
        "source_notebook_id": "",
        "source_workspace_id": "",
        "activation_reason": "Seed deterministic demo guardrail intent.",
        "activated_by": "example_pipeline_demo",
        "activated_at": now_utc,
        "superseded_by_rule_key": "",
        "notes": "Public-safe demo seed rule for the current metadata-driven workflow.",
        "approval_required": True,
        "approval_bypassed": False,
        "requires_governance_review": False,
        "requires_post_review": False,
        "governance_mode": "governed",
        "approval_policy": "governance_review",
        "submitted_by": "example_pipeline_demo",
        "submitted_at": now_utc,
        "reviewed_by": "example_pipeline_demo",
        "reviewed_at": now_utc,
        "review_decision": "approved",
        "review_comment": "Seeded for demo workflow.",
        "effective_from": now_utc,
        "_committed_at": now_utc,
        "_committed_by": "example_pipeline_demo",
        "_workspace_name": "",
        "_notebook_name": "example_pipeline_demo",
        "_metadata_lakehouse_name": CONFIG.path_config.paths[ENV]["metadata"].name,
        "_activity_id": "example_pipeline_demo",
    })

seeded_rule_rows = []
for source_table in DEMO_RULE_TABLES:
    if not source_table.startswith(DEMO_PREFIX):
        raise ValueError(f"Refusing to seed guardrail rules for non-demo table: {source_table}")
    seeded_rule_rows.extend(build_demo_rule_rows(source_table))

seeded_rule_df = spark.createDataFrame(seeded_rule_rows, schema=metadata_schemas["METADATA_GUARDRAIL_RULES"])
write_lakehouse_table(seeded_rule_df, CONFIG, ENV, "metadata", "METADATA_GUARDRAIL_RULES", schema=metadata_schema_name, mode="append")
display(seeded_rule_df.select("dataset_name", "table_name", "guardrail_type", "rule_type", "severity", "review_status").orderBy("table_name", "guardrail_type", "rule_type"))


DEMO_CATALOGUE_COLUMNS = {
    "demo_src_orders_happy": ["order_id", "customer_id", "order_date", "order_amount", "status", "ingestion_ts"],
    "demo_src_customers_happy": ["customer_id", "customer_name", "customer_segment"],
    "demo_unified_orders_enriched": ["order_id", "customer_id", "customer_name", "customer_segment", "order_date", "order_amount", "status", "ingestion_ts"],
}


def build_catalogue_rows(table_name, columns, stage):
    table_key = metadata_key(table_name)
    rows = [complete_row("METADATA_DATA_CATALOGUE", {
        "metadata_table_key": table_key,
        "metadata_column_key": "",
        "environment_name": ENV,
        "dataset_name": DEMO_PIPELINE_DATASET_NAME,
        "table_name": table_name,
        "column_name": "",
        "layer": "source" if stage == "source" else "unified",
        "asset_kind": "lakehouse",
        "pipeline_name": "demo_pipeline_orders",
        "profile_run_id": "example_pipeline_demo_seed",
        "profile_stage": stage,
        "profile_status": "success",
        "profiled_at": now_utc,
        "evidence_role": f"{stage}_profile",
        "row_count": scenario_frames.get(table_name).count() if table_name in scenario_frames else None,
        "profile_mode": "changing_data",
        "watermark_column": "ingestion_ts",
        "watermark_value": now_utc,
        "governance_mode": "governed",
        "approval_policy": "governance_review",
        "bypass_allowed": False,
        "policy_reason": "Demo metadata supports guardrail target selection before or after 02_pipeline profiling.",
        "policy_updated_by": "example_pipeline_demo",
        "policy_updated_at": now_utc,
        "_committed_at": now_utc,
        "_committed_by": "example_pipeline_demo",
        "_workspace_name": "",
        "_notebook_name": "example_pipeline_demo",
        "_metadata_lakehouse_name": CONFIG.path_config.paths[ENV]["metadata"].name,
        "_activity_id": "example_pipeline_demo",
    })]
    for column_name in columns:
        rows.append(complete_row("METADATA_DATA_CATALOGUE", {
            "metadata_table_key": table_key,
            "metadata_column_key": metadata_key(table_name, column_name),
            "environment_name": ENV,
            "dataset_name": DEMO_PIPELINE_DATASET_NAME,
            "table_name": table_name,
            "column_name": column_name,
            "layer": "source" if stage == "source" else "unified",
            "asset_kind": "lakehouse",
            "pipeline_name": "demo_pipeline_orders",
            "profile_run_id": "example_pipeline_demo_seed",
            "profile_stage": stage,
            "profile_status": "success",
            "profiled_at": now_utc,
            "evidence_role": f"{stage}_profile",
            "data_type": "string",
            "profile_mode": "changing_data",
            "watermark_column": "ingestion_ts",
            "watermark_value": now_utc,
            "governance_mode": "governed",
            "approval_policy": "governance_review",
            "bypass_allowed": False,
            "policy_reason": "Demo metadata supports guardrail target selection before or after 02_pipeline profiling.",
            "policy_updated_by": "example_pipeline_demo",
            "policy_updated_at": now_utc,
            "_committed_at": now_utc,
            "_committed_by": "example_pipeline_demo",
            "_workspace_name": "",
            "_notebook_name": "example_pipeline_demo",
            "_metadata_lakehouse_name": CONFIG.path_config.paths[ENV]["metadata"].name,
            "_activity_id": "example_pipeline_demo",
        }))
    return rows

catalogue_rows = []
for demo_table_name, columns in DEMO_CATALOGUE_COLUMNS.items():
    stage = "target" if demo_table_name.startswith("demo_unified_") else "source"
    catalogue_rows.extend(build_catalogue_rows(demo_table_name, columns, stage))

catalogue_df = spark.createDataFrame(catalogue_rows, schema=metadata_schemas["METADATA_DATA_CATALOGUE"])
write_lakehouse_table(catalogue_df, CONFIG, ENV, "metadata", "METADATA_DATA_CATALOGUE", schema=metadata_schema_name, mode="append")

result_row = complete_row("METADATA_GUARDRAIL_RESULTS", {
    "result_id": "example_pipeline_demo_seed__guardrail_results",
    "run_id": "example_pipeline_demo_seed",
    "rule_key": metadata_key("demo_src_orders_happy", "demo_pipeline_orders__demo_src_orders_happy__profile_behavior__changing_data"),
    "environment_name": ENV,
    "dataset_name": DEMO_PIPELINE_DATASET_NAME,
    "table_name": "demo_src_orders_happy",
    "column_name": "ingestion_ts",
    "guardrail_type": "profile_behavior",
    "rule_type": "changing_data",
    "status": "passed",
    "can_continue": True,
    "severity": "warning",
    "reason": "Seeded demo runtime evidence; 02_pipeline writes authoritative run results.",
    "expected_value_json": json.dumps({"profile_mode": "changing_data"}, sort_keys=True),
    "actual_value_json": json.dumps({"demo_seed": True}, sort_keys=True),
    "result_payload_json": json.dumps({"source_notebook_type": "example_pipeline_demo"}, sort_keys=True),
    "created_at": now_utc,
    "_committed_at": now_utc,
    "_committed_by": "example_pipeline_demo",
    "_workspace_name": "",
    "_notebook_name": "example_pipeline_demo",
    "_metadata_lakehouse_name": CONFIG.path_config.paths[ENV]["metadata"].name,
    "_activity_id": "example_pipeline_demo",
})
result_df = spark.createDataFrame([result_row], schema=metadata_schemas["METADATA_GUARDRAIL_RESULTS"])
write_lakehouse_table(result_df, CONFIG, ENV, "metadata", "METADATA_GUARDRAIL_RESULTS", schema=metadata_schema_name, mode="append")
display(catalogue_df.select("dataset_name", "table_name", "column_name", "profile_stage", "profile_status").orderBy("table_name", "column_name"))


StatementMeta(, , -1, Waiting, , Waiting, True)

## 6. Use these tables in `02_pipeline`

The default `02_pipeline` template reads the happy-path orders and customers tables directly:

```python
df_orders = read_lakehouse_table(CONFIG, ENV_NAME, "source", "demo_src_orders_happy", ...)
df_customers = read_lakehouse_table(CONFIG, ENV_NAME, "source", "demo_src_customers_happy", ...)
```

It then configures source guardrails, joins the source DataFrames, declares target tables after transformed DataFrames exist, runs target guardrails, writes targets, and records lineage/run-summary evidence.

For failure demos, update the relevant source table dictionary in `SOURCE_TABLES` after loading a replacement DataFrame. For the reload demo, run once with `demo_src_orders_reload_a`, then rerun with `demo_src_orders_reload_b` and compare profile behavior guardrail settings and physical write settings in the table configuration dictionaries.

The pipeline guardrail summary tables should show whether each guardrail passed, warned, failed, or blocked the target write. Catalogue, DQ, lineage, and run-summary evidence continues to use the metadata lakehouse configured by `00_env_config`.


## 7. Scenario catalogue

The catalogue below is the handoff checklist for selecting each demo source table in `02_pipeline`. All intended target table names are also prefixed with `demo_`.


In [ ]:
scenario_catalogue_rows = [
    {
        "scenario_name": "happy_path",
        "source_table": "demo_src_orders_happy + demo_src_customers_happy",
        "intended_target_table": "demo_unified_orders_enriched + demo_product_orders_summary",
        "guardrail_demonstrated": "schema + DQ + freshness + profile behavior pass",
        "expected_02_pipeline_result": "passes source and target guardrails, writes enriched and summary demo targets, records metadata evidence",
        "demo_notes": "Orders join to customers on customer_id; approved demo DQ rules should pass for the orders source.",
    },
    {
        "scenario_name": "schema_drift",
        "source_table": "demo_src_orders_schema_drift",
        "intended_target_table": "demo_unified_orders_schema_drift",
        "guardrail_demonstrated": "schema guardrail failure",
        "expected_02_pipeline_result": "source schema guardrail fails and blocks the target write",
        "demo_notes": "Table omits order_amount and adds promo_code; no DQ rules are seeded so schema drift is the visible guardrail.",
    },
    {
        "scenario_name": "dq_issue",
        "source_table": "demo_src_orders_dq_issue",
        "intended_target_table": "demo_unified_orders_dq_issue",
        "guardrail_demonstrated": "DQ guardrail failure",
        "expected_02_pipeline_result": "governance-approved DQ rules fail and block the target write",
        "demo_notes": "Includes null order_id, duplicate order_id, negative order_amount, and invalid status.",
    },
    {
        "scenario_name": "stale_source",
        "source_table": "demo_src_orders_stale",
        "intended_target_table": "demo_unified_orders_stale",
        "guardrail_demonstrated": "freshness guardrail failure",
        "expected_02_pipeline_result": "freshness guardrail fails and blocks the target write",
        "demo_notes": "Valid schema and DQ values, but order_date and ingestion_ts are intentionally old.",
    },
    {
        "scenario_name": "reload_a",
        "source_table": "demo_src_orders_reload_a",
        "intended_target_table": "demo_unified_orders_reload_demo",
        "guardrail_demonstrated": "profile behavior baseline / first run",
        "expected_02_pipeline_result": "establishes the first demo profile and target state for reload comparison",
        "demo_notes": "Run this first for the reload demo.",
    },
    {
        "scenario_name": "reload_b",
        "source_table": "demo_src_orders_reload_b",
        "intended_target_table": "demo_unified_orders_reload_demo",
        "guardrail_demonstrated": "profile behavior row-count and max-timestamp change",
        "expected_02_pipeline_result": "supports comparing static versus changing profile behavior in the real pipeline template",
        "demo_notes": "Run this second against the same target as reload_a, changing profile behavior and physical write variables as needed.",
    },
]

scenario_catalogue_df = spark.createDataFrame(scenario_catalogue_rows).select(
    "scenario_name",
    "source_table",
    "intended_target_table",
    "guardrail_demonstrated",
    "expected_02_pipeline_result",
    "demo_notes",
)
display(scenario_catalogue_df.orderBy("scenario_name"))


StatementMeta(, , -1, Waiting, , Waiting, True)